# LoRA: run OK check, smoke run, LR sweep, final runs

One notebook for both LoRA ranks. **Eddie: `RANK = 4`. Dylan: `RANK = 32`** (cell 3). Everything else is the same.

Order (Paddy's 14 Oct list):
1. Setup (T4, Drive, latest code, images into `data/raw`)
2. `python -m src.run` self-check, must end with **run OK**
3. Settings
4. Smoke run, timed. It *is* the first sweep run (lr 1e-3, seed 0), so it isn't wasted
5. LR sweep, validation only: seeds 0–1 first (the minimum), then 2–4 if time allows
6. Summary: pick the lowest mean val CE, flag grid edge / 50-epoch cap, compare seed 2 with frozen, print a decision-log draft
7. Final runs (seeds 0–4) — only after the chosen lr is committed in `configs/r<RANK>.yaml`

Sweep runs go to `runs/pilot/lora_lr/` and `results/pilot_lora_lr_r<RANK>.csv` on Drive, never `runs/` or `runs.csv`.
The test set is never loaded.

**If Colab disconnects:** rerun cells 1 and 3, then the same cell you were on. Finished runs are skipped, a half-done run resumes from `last.pt`.

In [ ]:
# 1. Setup (~3-5 min). First: Runtime -> Change runtime type -> T4 GPU
import os
from google.colab import drive
drive.mount('/content/drive')
DRIVE = "/content/drive/MyDrive/aai-project"
REPO = "/content/Advanced-AI"

if not os.path.exists(f"{REPO}/.git"):
    !rm -rf {REPO}
    from google.colab import userdata
    TOKEN = userdata.get("GH_TOKEN").strip()
    !git clone -q https://{TOKEN}@github.com/dylan-teehan-skc/Advanced-AI.git {REPO}
assert os.path.exists(f"{REPO}/.git"), "clone failed: check the GH_TOKEN secret"
%cd {REPO}
!git pull -q
!git log --oneline -1
!pip install -q -r requirements.txt
!pip uninstall -y -q torchao

# the self-check and run() read images from data/raw/EuroSAT_RGB inside the repo (Paddy's note)
if not os.path.exists("data/raw/EuroSAT_RGB"):
    !cp "{DRIVE}/EuroSAT_RGB.zip" /content/
    !unzip -q /content/EuroSAT_RGB.zip -d data/raw
!ls data/raw/EuroSAT_RGB
!nvidia-smi --query-gpu=name --format=csv,noheader

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/Advanced-AI
36417ad (HEAD -> main, origin/main, origin/HEAD) Decision log: metrics entry
AnnualCrop  HerbaceousVegetation  Industrial  PermanentCrop  River
Forest	    Highway		  Pasture     Residential    SeaLake
Tesla T4


In [ ]:
# 2. Self-check (~2-4 min, CPU, downloads the ViT once). Last line must be: run OK
!python -m src.run

config.json: 100% 502/502 [00:00<00:00, 2.02MB/s]

model.safetensors: downloading bytes:  17% 58.5M/346M [00:01<00:05, 49.4MB/s, 3.28MB/s  ]
model.safetensors: downloading bytes:  74% 256M/346M [00:02<00:00, 215MB/s, 21.0MB/s  ]
model.safetensors: reconstructing file:  11% 38.5M/346M [00:02<00:24, 12.7MB/s, 3.44MB/s  ]
model.safetensors: downloading bytes:  87% 301M/346M [00:02<00:00, 215MB/s, 24.3MB/s  ]
model.safetensors: downloading bytes:  95% 328M/346M [00:02<00:00, 226MB/s, 26.8MB/s  ]
model.safetensors: reconstructing file:  43% 149M/346M [00:03<00:01, 104MB/s, 8.34MB/s  ]  
model.safetensors: reconstructing file:  59% 204M/346M [00:03<00:00, 161MB/s, 13.5MB/s  ]
model.safetensors: reconstructing file:  91% 315M/346M [00:05<00:00, 65.2MB/s, 18.4MB/s  ]
model.safetensors: downloading bytes: 100% 328M/328M [00:05<00:00, 55.0MB/s, 29.0MB/s  ]
model.safetensors: reconstructing file: 100% 346M/346M [00:05<00:00, 58.0MB/s, 26.9MB/s  ]
Loading weights: 100% 198/198 [00:00<00:00, 5628.6

In [ ]:
# 3. Settings. Rerun this after any disconnect.
import time
from pathlib import Path

import pandas as pd
import torch
import yaml

from src.run import load_config, run

RANK = 32                      # Eddie 4, Dylan 32
COND = f"r{RANK}"
LRS = [1e-4, 3e-4, 1e-3]      # Paddy's grid
SWEEP_ROOT = Path(DRIVE) / "runs/pilot/lora_lr"
SWEEP_CSV = f"{DRIVE}/results/pilot_lora_lr_{COND}.csv"
FINAL_CSV = f"{DRIVE}/results/runs_{COND}.csv"   # own CSV per rank, so two accounts never append to one file

base = load_config(f"configs/{COND}.yaml")
assert base["n_per_class"] == 20, "old config: git pull didn't bring in Paddy's N = 20 commit"
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4")


def lr_folder(lr):
    return SWEEP_ROOT / f"{COND}_lr{lr:g}"


def sweep_config(lr):
    """configs/r<RANK>.yaml with only lr changed, in its own folder so runs at different lrs never collide."""
    folder = lr_folder(lr)
    folder.mkdir(parents=True, exist_ok=True)
    (folder / "config.yaml").write_text(yaml.safe_dump({**base, "lr": lr}))
    return folder / "config.yaml", folder


def epochs_run(folder, seed):
    """How many epochs the run actually trained (early stop or the 50 cap), from its last checkpoint."""
    return torch.load(Path(folder) / f"{COND}_s{seed}" / "last.pt", map_location="cpu")["state"]["epoch"]


# a leftover folder in runs/ would be skipped (row.json) or resumed (last.pt) instead of run fresh
stale = sorted(p.name for p in (Path(DRIVE) / "runs").glob(f"{COND}_s*"))
print(f"Already in runs/: {stale}. Fine if they are your final runs resuming; otherwise move them out before cell 7"
      if stale else "runs/ is clear for the final runs")
old = sorted(str(p.relative_to(SWEEP_ROOT)) for p in SWEEP_ROOT.glob(f"{COND}_lr*/{COND}_s*"))
print(f"sweep runs already on Drive (will be skipped or resumed): {old}" if old else "no sweep runs yet")

GPU: Tesla T4
runs/ is clear for the final runs
sweep runs already on Drive (will be skipped or resumed): ['r32_lr0.0001/r32_s0', 'r32_lr0.0001/r32_s1', 'r32_lr0.0001/r32_s2', 'r32_lr0.0001/r32_s3', 'r32_lr0.0001/r32_s4', 'r32_lr0.0003/r32_s0', 'r32_lr0.0003/r32_s1', 'r32_lr0.0003/r32_s2', 'r32_lr0.0003/r32_s3', 'r32_lr0.001 (1)/r32_s0', 'r32_lr0.001 (2)/r32_s0', 'r32_lr0.001 (3)/r32_s0', 'r32_lr0.001/r32_s0', 'r32_lr0.001/r32_s1', 'r32_lr0.001/r32_s2', 'r32_lr0.001/r32_s3']


In [ ]:
# 4. Smoke run on split_v1, timed. Same settings as sweep run lr 1e-3 / seed 0, so it counts towards the sweep.
cfg, folder = sweep_config(1e-3)
row = run(cfg, seed=0, out_root=folder, results_csv=SWEEP_CSV)
ep = epochs_run(folder, 0)
mins = row["train_time_s"] / 60
print(f"\nsmoke run: {mins:.1f} min training, {ep} epochs ({row['train_time_s'] / ep:.0f} s/epoch), "
      f"best epoch {row['best_epoch']}, val CE {row['val_ce']:.4f}, balanced acc {row['val_bal_acc']:.3f}, "
      f"{row['trainable_params']:,} trainable")
print(f"rough budget if every run is like this one (+~1 min each to load and score):\n"
      f"  full sweep (14 more) + 5 finals: {19 * (mins + 1) / 60:.1f} h\n"
      f"  seeds 0-1 sweep (5 more) + 5 finals: {10 * (mins + 1) / 60:.1f} h")
print("Lower lrs usually run more epochs, so treat this as a floor.")

r32_s0 already finished, skipping

smoke run: 17.9 min training, 20 epochs (54 s/epoch), best epoch 15, val CE 0.3020, balanced acc 0.902, 1,187,338 trainable
rough budget if every run is like this one (+~1 min each to load and score):
  full sweep (14 more) + 5 finals: 6.0 h
  seeds 0-1 sweep (5 more) + 5 finals: 3.1 h
Lower lrs usually run more epochs, so treat this as a floor.


In [ ]:
# 5a. Sweep, seeds 0-1 for every lr (the minimum). One seed at a time across all lrs,
#     so if Colab cuts us off, every lr has the same seeds.
def sweep(seeds):
    for seed in seeds:
        for lr in LRS:
            cfg, folder = sweep_config(lr)
            r = run(cfg, seed=seed, out_root=folder, results_csv=SWEEP_CSV)
            print(f"== lr {lr:g} seed {seed}: val CE {r['val_ce']:.4f}, balanced acc {r['val_bal_acc']:.3f}, "
                  f"best epoch {r['best_epoch']} of {epochs_run(folder, seed)}, {r['train_time_s'] / 60:.1f} min\n")


sweep([0, 1])

r32_s0 already finished, skipping
== lr 0.0001 seed 0: val CE 0.5450, balanced acc 0.893, best epoch 50 of 50, 46.0 min

r32_s0 already finished, skipping
== lr 0.0003 seed 0: val CE 0.3750, balanced acc 0.896, best epoch 27 of 32, 29.7 min

r32_s0 already finished, skipping
== lr 0.001 seed 0: val CE 0.3020, balanced acc 0.902, best epoch 15 of 20, 17.9 min

r32_s1 already finished, skipping
== lr 0.0001 seed 1: val CE 0.5786, balanced acc 0.878, best epoch 49 of 50, 47.2 min

r32_s1 already finished, skipping
== lr 0.0003 seed 1: val CE 0.3026, balanced acc 0.898, best epoch 46 of 50, 47.2 min

r32_s1 already finished, skipping
== lr 0.001 seed 1: val CE 0.2825, balanced acc 0.905, best epoch 19 of 24, 22.6 min



In [ ]:
# 5b. Seeds 2-4, to match frozen's 5 seeds per value. Skip if time is short, and say so in the log.
sweep([2, 3, 4])

r32_s2 already finished, skipping
== lr 0.0001 seed 2: val CE 0.5883, balanced acc 0.869, best epoch 50 of 50, 47.2 min

r32_s2 already finished, skipping
== lr 0.0003 seed 2: val CE 0.3351, balanced acc 0.894, best epoch 43 of 48, 45.9 min

r32_s2 already finished, skipping
== lr 0.001 seed 2: val CE 0.3030, balanced acc 0.902, best epoch 18 of 23, 22.2 min

r32_s3 already finished, skipping
== lr 0.0001 seed 3: val CE 0.5705, balanced acc 0.872, best epoch 50 of 50, 48.2 min

r32_s3 already finished, skipping
== lr 0.0003 seed 3: val CE 0.3199, balanced acc 0.899, best epoch 40 of 45, 43.4 min

r32_s3 already finished, skipping
== lr 0.001 seed 3: val CE 0.2960, balanced acc 0.908, best epoch 19 of 24, 23.1 min



Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.bias   | UNEXPECTED | 
pooler.dense.weight | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


rank 32: 1,187,338 trainable parameters (head 7,690 + LoRA 1,179,648)
resumed r32_s4 after epoch 27
epoch 28: train CE 0.9794, val CE 1.1276  (best)
epoch 29: train CE 0.9284, val CE 1.0829  (best)
epoch 30: train CE 0.8779, val CE 1.0374  (best)
epoch 31: train CE 0.8356, val CE 0.9996  (best)
epoch 32: train CE 0.7877, val CE 0.9677  (best)
epoch 33: train CE 0.7448, val CE 0.9148  (best)
epoch 34: train CE 0.7077, val CE 0.8935  (best)
epoch 35: train CE 0.6613, val CE 0.8489  (best)
epoch 36: train CE 0.6245, val CE 0.8231  (best)
epoch 37: train CE 0.5974, val CE 0.7853  (best)
epoch 38: train CE 0.5490, val CE 0.7616  (best)
epoch 39: train CE 0.5249, val CE 0.7343  (best)
epoch 40: train CE 0.4941, val CE 0.7035  (best)
epoch 41: train CE 0.4607, val CE 0.6797  (best)
epoch 42: train CE 0.4369, val CE 0.6592  (best)
epoch 43: train CE 0.4017, val CE 0.6398  (best)
epoch 44: train CE 0.3892, val CE 0.6222  (best)
epoch 45: train CE 0.3641, val CE 0.5971  (best)
epoch 46: train CE

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.bias   | UNEXPECTED | 
pooler.dense.weight | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


rank 32: 1,187,338 trainable parameters (head 7,690 + LoRA 1,179,648)
epoch 1: train CE 2.2969, val CE 2.2588  (best)
epoch 2: train CE 2.2205, val CE 2.1981  (best)
epoch 3: train CE 2.1433, val CE 2.1274  (best)
epoch 4: train CE 2.0498, val CE 2.0406  (best)
epoch 5: train CE 1.9386, val CE 1.9366  (best)
epoch 6: train CE 1.8203, val CE 1.8212  (best)
epoch 7: train CE 1.6829, val CE 1.6899  (best)
epoch 8: train CE 1.5353, val CE 1.5591  (best)
epoch 9: train CE 1.3779, val CE 1.4203  (best)
epoch 10: train CE 1.2311, val CE 1.3042  (best)
epoch 11: train CE 1.0827, val CE 1.1526  (best)
epoch 12: train CE 0.9397, val CE 1.0425  (best)
epoch 13: train CE 0.8065, val CE 0.9302  (best)
epoch 14: train CE 0.7009, val CE 0.8189  (best)
epoch 15: train CE 0.6010, val CE 0.7341  (best)
epoch 16: train CE 0.4948, val CE 0.6686  (best)
epoch 17: train CE 0.4274, val CE 0.5975  (best)
epoch 18: train CE 0.3566, val CE 0.5679  (best)
epoch 19: train CE 0.3034, val CE 0.5035  (best)
epoch 20

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.bias   | UNEXPECTED | 
pooler.dense.weight | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


rank 32: 1,187,338 trainable parameters (head 7,690 + LoRA 1,179,648)
epoch 1: train CE 2.2519, val CE 2.1195  (best)
epoch 2: train CE 1.9799, val CE 1.8500  (best)
epoch 3: train CE 1.6650, val CE 1.5538  (best)
epoch 4: train CE 1.3206, val CE 1.2573  (best)
epoch 5: train CE 0.9897, val CE 0.9470  (best)
epoch 6: train CE 0.7020, val CE 0.7324  (best)
epoch 7: train CE 0.4707, val CE 0.5552  (best)
epoch 8: train CE 0.3072, val CE 0.4398  (best)
epoch 9: train CE 0.1906, val CE 0.3988  (best)
epoch 10: train CE 0.1284, val CE 0.3298  (best)
epoch 11: train CE 0.0927, val CE 0.3148  (best)
epoch 12: train CE 0.0606, val CE 0.3337
epoch 13: train CE 0.0471, val CE 0.2836  (best)
epoch 14: train CE 0.0381, val CE 0.2826  (best)
epoch 15: train CE 0.0303, val CE 0.2751  (best)
epoch 16: train CE 0.0260, val CE 0.2588  (best)
epoch 17: train CE 0.0221, val CE 0.2639
epoch 18: train CE 0.0195, val CE 0.2693
epoch 19: train CE 0.0175, val CE 0.2716
epoch 20: train CE 0.0158, val CE 0.2700

In [ ]:
# 6. Summary and pick. Rule: lowest mean validation CE, over the seeds every lr has finished.
df = pd.read_csv(SWEEP_CSV)
df = df[df["condition"] == COND].drop_duplicates(["lr", "seed"], keep="last")
seeds = sorted(set.intersection(*(set(g["seed"]) for _, g in df.groupby("lr"))))
missing = set(LRS) - set(df["lr"])
assert not missing, f"no finished runs yet for lr {missing}"
df = df[df["seed"].isin(seeds)].copy()
df["epochs"] = [epochs_run(lr_folder(lr), s) for lr, s in zip(df["lr"], df["seed"])]

t = df.groupby("lr").agg(runs=("seed", "count"), ce=("val_ce", "mean"), ce_sd=("val_ce", "std"),
                         acc=("val_bal_acc", "mean"), acc_sd=("val_bal_acc", "std"),
                         best_epochs=("best_epoch", list), epochs=("epochs", list),
                         mins=("train_time_s", lambda s: round(s.mean() / 60, 1))).reset_index()
print(f"{COND}, N = 20, seeds {seeds}\n")
print(t.round(4).to_string(index=False))

best = t.loc[t["ce"].idxmin()]
CHOSEN_LR = float(best["lr"])
print(f"\nRule: lowest mean val CE -> lr = {CHOSEN_LR:g}")
flags = []
if CHOSEN_LR in (min(LRS), max(LRS)):
    flags.append(f"the best lr ({CHOSEN_LR:g}) is at the edge of the grid; the true best may lie outside it")
cap = base["max_epochs"]
hit = df[df["epochs"] >= cap]
if len(hit):
    flags.append("hit the 50-epoch cap: " + ", ".join(f"lr {r.lr:g} s{r.seed} (best epoch {r.best_epoch})"
                                                       for r in hit.itertuples()))
if len(seeds) < 5:
    flags.append(f"only seeds {seeds} for every lr (frozen used 5)")
for f in flags:
    print("FLAG:", f)

# seed 2 was frozen's weak seed (0.605); is it weak for LoRA too?
frozen = pd.read_csv("results/runs.csv").query("condition == 'frozen'").set_index("seed")["val_ce"]
mine = df[df["lr"] == CHOSEN_LR].set_index("seed")["val_ce"]
print("\nval CE per seed, frozen vs this rank at the chosen lr:")
print(pd.DataFrame({"frozen": frozen, f"{COND} lr {CHOSEN_LR:g}": mine}).round(3).to_string())

# draft decision-log entry (Paddy's format). Check it, edit the Why if needed, then paste into your log.
per_lr = "; ".join(f"lr {r.lr:g} gave CE {r.ce:.3f} +/- {r.ce_sd:.3f}" for r in t.itertuples())
print(f"""
---- draft for docs/decision-log ----
## {time.strftime('%-d %b %Y')} - {COND} learning rate = {CHOSEN_LR:g}
- **Decision:** {COND} uses lr = {CHOSEN_LR:g}, chosen by the sweep rule: lowest mean validation CE at N = 20 over 1e-4, 3e-4, 1e-3.
- **Why:** {len(seeds)} seeds each: {per_lr}.{' ' + '. '.join(f[0].upper() + f[1:] for f in flags) + '.' if flags else ' The best is inside the grid and no run hit the 50-epoch cap.'} Results in `results/pilot_lora_lr_{COND}.csv`.
- **Affects:** `lr` in `configs/{COND}.yaml`, the final {COND} runs.""")

r32, N = 20, seeds [2, 3, 4]

    lr  runs     ce  ce_sd    acc  acc_sd  best_epochs       epochs  mins
0.0001     3 0.5623 0.0309 0.8786  0.0140 [50, 50, 50] [50, 50, 50]  49.1
0.0003     3 0.3212 0.0132 0.9005  0.0078 [43, 40, 41] [48, 45, 46]  46.5
0.0010     3 0.2859 0.0237 0.9089  0.0075 [18, 19, 16] [23, 24, 21]  22.8

Rule: lowest mean val CE -> lr = 0.001
FLAG: the best lr (0.001) is at the edge of the grid; the true best may lie outside it
FLAG: hit the 50-epoch cap: lr 0.0001 s2 (best epoch 50), lr 0.0001 s3 (best epoch 50), lr 0.0001 s4 (best epoch 50)
FLAG: only seeds [2, 3, 4] for every lr (frozen used 5)

val CE per seed, frozen vs this rank at the chosen lr:
      frozen  r32 lr 0.001
seed                      
0      0.547           NaN
1      0.550           NaN
2      0.605         0.303
3      0.546         0.296
4      0.547         0.259

---- draft for docs/decision-log ----
## 19 Oct 2026 - r32 learning rate = 0.001
- **Decision:** r32 uses lr = 0.001, chosen by 

### Before cell 7 (on your laptop, ~5 min)
1. `git pull`
2. In `configs/r<RANK>.yaml` set `lr:` to the chosen value **written with a dot and a signed exponent**, e.g. `3.0e-4`. PyYAML reads `3e-4` as a *string*, not a number.
3. Paste the decision-log draft into your log, copy `pilot_lora_lr_r<RANK>.csv` from Drive `aai-project/results/` into the repo's `results/`.
4. Commit and push. Then run cell 7 (it pulls first, so each row records the commit with the chosen lr).

In [ ]:
# 7. Final runs: seeds 0-4 at the chosen lr into runs/ and runs_r<RANK>.csv. Validation only.
!git pull -q
!git log --oneline -1
CHOSEN_LR = globals().get("CHOSEN_LR")  # after a restart, type it instead, e.g. CHOSEN_LR = 3e-4
assert CHOSEN_LR is not None, "set CHOSEN_LR (cell 6 sets it, or type it)"
cfg = load_config(f"configs/{COND}.yaml")
assert isinstance(cfg["lr"], float), f"lr in configs/{COND}.yaml read as {cfg['lr']!r}; write it like 3.0e-4"
assert abs(cfg["lr"] - CHOSEN_LR) < 1e-12, f"configs/{COND}.yaml still has lr {cfg['lr']:g}; commit the chosen lr and push first"

for seed in range(5):
    r = run(f"configs/{COND}.yaml", seed=seed, out_root=f"{DRIVE}/runs", results_csv=FINAL_CSV)
    print(f"== {r['run_id']}: val CE {r['val_ce']:.4f}, balanced acc {r['val_bal_acc']:.3f}, best epoch {r['best_epoch']}\n")

final = pd.read_csv(FINAL_CSV)
print(final[["run_id", "lr", "best_epoch", "val_ce", "val_bal_acc", "train_time_s", "gpu", "git_commit"]].to_string(index=False))
print(f"\nmean val CE {final['val_ce'].mean():.3f} +/- {final['val_ce'].std():.3f}, "
      f"balanced acc {final['val_bal_acc'].mean():.1%}   (frozen: 0.559 +/- 0.026, 80.2%)")

36417ad (HEAD -> main, origin/main, origin/HEAD) Decision log: metrics entry


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.bias   | UNEXPECTED | 
pooler.dense.weight | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


rank 32: 1,187,338 trainable parameters (head 7,690 + LoRA 1,179,648)
epoch 1: train CE 2.2644, val CE 2.1219  (best)
epoch 2: train CE 1.9909, val CE 1.8454  (best)
epoch 3: train CE 1.6317, val CE 1.4863  (best)
epoch 4: train CE 1.2774, val CE 1.2099  (best)
epoch 5: train CE 0.9340, val CE 0.9414  (best)
epoch 6: train CE 0.6764, val CE 0.7167  (best)
epoch 7: train CE 0.4793, val CE 0.6383  (best)
epoch 8: train CE 0.3256, val CE 0.4915  (best)
epoch 9: train CE 0.2244, val CE 0.4226  (best)
epoch 10: train CE 0.1933, val CE 0.4570
epoch 11: train CE 0.1089, val CE 0.3770  (best)
epoch 12: train CE 0.0774, val CE 0.3749  (best)
epoch 13: train CE 0.0514, val CE 0.3697  (best)
epoch 14: train CE 0.0405, val CE 0.3167  (best)
epoch 15: train CE 0.0324, val CE 0.3103  (best)
epoch 16: train CE 0.0272, val CE 0.3091  (best)
epoch 17: train CE 0.0233, val CE 0.3220
epoch 18: train CE 0.0207, val CE 0.3225
epoch 19: train CE 0.0184, val CE 0.3229
epoch 20: train CE 0.0168, val CE 0.3258

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

...

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.bias   | UNEXPECTED | 
pooler.dense.weight | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


rank 32: 1,187,338 trainable parameters (head 7,690 + LoRA 1,179,648)
epoch 1: train CE 2.2534, val CE 2.1274  (best)
epoch 2: train CE 1.9706, val CE 1.8275  (best)
epoch 3: train CE 1.6384, val CE 1.5079  (best)
epoch 4: train CE 1.3022, val CE 1.2217  (best)
epoch 5: train CE 0.9800, val CE 0.9653  (best)
epoch 6: train CE 0.7151, val CE 0.7562  (best)
epoch 7: train CE 0.4798, val CE 0.6177  (best)
epoch 8: train CE 0.3369, val CE 0.5342  (best)
epoch 9: train CE 0.2285, val CE 0.4151  (best)
epoch 10: train CE 0.1529, val CE 0.4705
epoch 11: train CE 0.0958, val CE 0.3644  (best)
epoch 12: train CE 0.0717, val CE 0.3269  (best)
epoch 13: train CE 0.0564, val CE 0.3214  (best)
epoch 14: train CE 0.0558, val CE 0.3097  (best)
epoch 15: train CE 0.0450, val CE 0.3789
epoch 16: train CE 0.0575, val CE 0.3984
epoch 17: train CE 0.0289, val CE 0.4317
epoch 18: train CE 0.0254, val CE 0.3707
epoch 19: train CE 0.0215, val CE 0.3186
== r32_s3: val CE 0.3097, balanced acc 0.902, best epoch

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.bias   | UNEXPECTED | 
pooler.dense.weight | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


rank 32: 1,187,338 trainable parameters (head 7,690 + LoRA 1,179,648)
epoch 1: train CE 2.2519, val CE 2.1195  (best)
epoch 2: train CE 1.9799, val CE 1.8500  (best)
epoch 3: train CE 1.6650, val CE 1.5542  (best)
epoch 4: train CE 1.3199, val CE 1.2579  (best)
epoch 5: train CE 0.9878, val CE 0.9450  (best)
epoch 6: train CE 0.6994, val CE 0.7302  (best)
epoch 7: train CE 0.4714, val CE 0.5565  (best)
epoch 8: train CE 0.3081, val CE 0.4485  (best)
epoch 9: train CE 0.1925, val CE 0.4044  (best)
epoch 10: train CE 0.1324, val CE 0.3347  (best)
epoch 11: train CE 0.0920, val CE 0.3251  (best)
epoch 12: train CE 0.0670, val CE 0.3128  (best)
epoch 13: train CE 0.0489, val CE 0.3278
epoch 14: train CE 0.0409, val CE 0.2914  (best)
epoch 15: train CE 0.0306, val CE 0.2936
epoch 16: train CE 0.0261, val CE 0.2906  (best)
epoch 17: train CE 0.0225, val CE 0.2868  (best)
epoch 18: train CE 0.0197, val CE 0.2831  (best)
epoch 19: train CE 0.0174, val CE 0.2820  (best)
epoch 20: train CE 0.015